# Colab Fine-tune YOLOv8 Weapon Detection
# Cell 1: Notebook metadata and purpose
"""
This notebook prepares Google Colab to fine-tune the YOLOv8 weapon detection model.
Sections:
1. Mount Drive
2. Install dependencies
3. Prepare repo/dataset
4. Copy weights and set params
5. Train (resume or fresh)
6. Evaluate and save weights to Drive
"""

In [ ]:
# Cell 2: Print environment info
import sys, os
print('Python', sys.version)
print('Working dir:', os.getcwd())


In [ ]:
# Cell 3: Mount Google Drive
from google.colab import drive
print('Mounting Google Drive...')
drive.mount('/content/drive')


In [ ]:
# Cell 4: Install dependencies
# ultralytics installs a compatible torch on Colab
!pip install -q --upgrade pip
!pip install -q ultralytics==8.* facenet-pytorch mediapipe opencv-python-headless

# Show installed ultralytics version
python -c "import ultralytics, sys; print('ultralytics', ultralytics.__version__)"


In [ ]:
# Cell 5: Prepare repo & dataset
# Option A: Clone from GitHub (if you pushed your repo)
# !git clone https://github.com/<your-user>/Threat-Detection.git /content/repo
# %cd /content/repo

# Option B: Use files from Drive (recommended for private repos)
# Assume user uploaded a zip or copied project into Drive path below
DRIVE_ROOT = '/content/drive/MyDrive/Threat-Detection'

import os
if os.path.exists(DRIVE_ROOT):
    print('Using repo from Drive:', DRIVE_ROOT)
    %cd $DRIVE_ROOT
else:
    print('Drive repo not found. Please upload your repo to Drive or clone from GitHub and re-run this cell.')

# Show dataset tree
!ls -la
!ls -la Weapon\ 2.v2i.yolov8 || true


In [ ]:
# Cell 6: Copy weights (from Drive) into workspace
# Place your last.pt or best.pt in Drive at DRIVE_ROOT/runs/train/weapon_train3/weights/last.pt
import shutil
weights_src = '/content/drive/MyDrive/Threat-Detection/runs/train/weapon_train3/weights/last.pt'
weights_dst = '/content/drive/MyDrive/Threat-Detection/camera_detection/models/weapon_best_resume.pt'
if os.path.exists(weights_src):
    print('Copying', weights_src, '->', weights_dst)
    os.makedirs(os.path.dirname(weights_dst), exist_ok=True)
    shutil.copy(weights_src, weights_dst)
    print('Copied.')
else:
    print('No checkpoint found at', weights_src, '\nYou can upload last.pt to Drive or start fresh from yolov8n.pt')


In [ ]:
# Cell 7: Training cell (ultralytics CLI)
# Edit args below as needed: epochs, imgsz, batch
WEIGHTS = '/content/drive/MyDrive/Threat-Detection/camera_detection/models/weapon_best_resume.pt'  # or 'yolov8n.pt'
DATA_YAML = '/content/drive/MyDrive/Threat-Detection/data/weapon_data.yaml'
EPOCHS = 50
IMGSZ = 640
BATCH = 16
PROJECT = '/content/drive/MyDrive/Threat-Detection/runs'
NAME = 'weapon_finetune_colab'

print('Training with', WEIGHTS)
# Use ultralytics CLI
get_ipython().system_raw(f"yolo task=detect mode=train data={DATA_YAML} model={WEIGHTS} epochs={EPOCHS} imgsz={IMGSZ} batch={BATCH} project={PROJECT} name={NAME} cache=True &")
print('Training started in background (see runtime output).')


In [ ]:
# Cell 8: Monitor training (tail ultralytics log)
# If training started in background above, you can stream logs:
import time
log_path = '/content/drive/MyDrive/Threat-Detection/runs/weapon_finetune_colab/train_log.txt'
print('If you used background training, open the run directory in Drive. Otherwise, logs appear in the cell output.')


In [ ]:
# Cell 9: Evaluate trained model on test set
# After training completes, point to the final weights and run validation
WEIGHTS_FINAL = '/content/drive/MyDrive/Threat-Detection/runs/weapon_finetune_colab/weights/best.pt'
DATA_YAML = '/content/drive/MyDrive/Threat-Detection/data/weapon_data.yaml'

from ultralytics import YOLO
print('Evaluating', WEIGHTS_FINAL)
model = YOLO(WEIGHTS_FINAL)
res = model.val(data=DATA_YAML, imgsz=640)
print(res)


In [ ]:
# Cell 10: Download weights to local machine (optional)
from google.colab import files
weights_to_download = '/content/drive/MyDrive/Threat-Detection/runs/weapon_finetune_colab/weights/best.pt'
if os.path.exists(weights_to_download):
    print('Downloading', weights_to_download)
    files.download(weights_to_download)
else:
    print('Weights not found at', weights_to_download)
